# Order Book Viewer

Depth-5 L2 order-book view using `tools.lob_viewer.plot_order_book`, L2 depth parquet files in `data/orderbook_l2_parquet`, and L3/MBO event parquet files in `data/databento_glbx_mdp3_mbo_full_day_parquet`.

The defaults use a short ES RTH-open window. Styling follows common trading-screen convention: dark background, bid levels in green, ask levels in red/orange, small cross markers at every displayed book update, and L3 events as directional markers.


In [1]:
from pathlib import Path
import sys

import importlib

import polars as pl
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tools").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tools.viewer as viewer_mod
import tools.lob_viewer as lob_viewer_mod
importlib.reload(viewer_mod)
importlib.reload(lob_viewer_mod)
from tools.lob_viewer import INVALID_PRICE, PRICE_SCALE, plot_order_book
from tools.viewer import add_x_pan_buttons


In [2]:
try:
    import plotly  # noqa: F401
    import plotly.io as pio
    pio.renderers.default = "plotly_mimetype+notebook_connected"
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs Plotly because tools.viewer imports it lazily when rendering. "
        "Install plotly in the active notebook environment, then rerun."
    ) from exc

## Window And Instrument

`plot_order_book(timezone=...)` interprets naive `start`/`end` strings in the display timezone, then filters the UTC source column efficiently before rendering.


In [3]:
L2_DIR = PROJECT_ROOT / "data" / "orderbook_l2_parquet"
L3_DIR = PROJECT_ROOT / "data" / "databento_glbx_mdp3_mbo_full_day_parquet"

PRODUCT = "ESM6"
SESSION_DATE = "2026-05-06"
DISPLAY_TZ = "America/New_York"

# Keep this tight for notebook interactivity. Expand after the first render feels good.
START = f"{SESSION_DATE} 09:30:00.8"
END = f"{SESSION_DATE} 09:30:01.2"

DEPTH_LEVELS = 1
MAX_POINTS_PER_SERIES = 15_000

l2_matches = sorted(L2_DIR.glob(f"{PRODUCT}_{SESSION_DATE}_*_l2_d5.parquet"))
if not l2_matches:
    raise FileNotFoundError(f"No L2 parquet matched {PRODUCT}_{SESSION_DATE}_*_l2_d5.parquet in {L2_DIR}")
BOOK_PATH = l2_matches[0]

l3_matches = sorted(L3_DIR.glob(f"{PRODUCT}_{SESSION_DATE}_*_full_day.parquet"))
if not l3_matches:
    raise FileNotFoundError(f"No L3 parquet matched {PRODUCT}_{SESSION_DATE}_*_full_day.parquet in {L3_DIR}")
L3_PATH = l3_matches[0]

BOOK_PATH, L3_PATH


FileNotFoundError: No L3 parquet matched ESM6_2026-05-06_*_full_day.parquet in /home/jli/projects/rep/data/databento_glbx_mdp3_mbo_full_day_parquet

In [ ]:
raw_l2 = pl.scan_parquet(BOOK_PATH)
raw_l3 = pl.scan_parquet(L3_PATH)

{
    "l2": raw_l2.collect_schema(),
    "l3": raw_l3.collect_schema(),
}


## Lazy Source

Databento-style prices are fixed-point integers. `plot_order_book` keeps raw L2/L3 parquet columns as hover data and decodes prices only for display.


In [ ]:
def valid_price(px_col: str, sz_col: str | None = None) -> pl.Expr:
    valid_px = (pl.col(px_col) > 0) & (pl.col(px_col) < INVALID_PRICE)
    if sz_col is not None:
        valid_px = valid_px & (pl.col(sz_col) > 0)
    return valid_px


def display_price(px_col: str, sz_col: str | None = None) -> pl.Expr:
    return pl.when(valid_price(px_col, sz_col)).then(pl.col(px_col) / PRICE_SCALE).otherwise(None)


book = pl.scan_parquet(BOOK_PATH)
l3 = pl.scan_parquet(L3_PATH)


In [ ]:
session_year, session_month, session_day = map(int, SESSION_DATE.split("-"))

l2_summary = book.filter(
    pl.col("ts_event").dt.convert_time_zone(DISPLAY_TZ).is_between(
        pl.datetime(session_year, session_month, session_day, 9, 30, time_zone=DISPLAY_TZ),
        pl.datetime(session_year, session_month, session_day, 9, 30, 5, time_zone=DISPLAY_TZ),
    )
).select(
    pl.len().alias("rows"),
    valid_price("trade_px", "trade_sz").sum().alias("trade_rows"),
    valid_price("bid_px_0", "bid_sz_0").sum().alias("valid_l1_bid_rows"),
    valid_price("ask_px_0", "ask_sz_0").sum().alias("valid_l1_ask_rows"),
    display_price("bid_px_0", "bid_sz_0").median().alias("median_bid"),
    display_price("ask_px_0", "ask_sz_0").median().alias("median_ask"),
).collect()

l3_summary = (
    l3.filter(
        pl.col("ts_event").dt.convert_time_zone(DISPLAY_TZ).is_between(
            pl.datetime(session_year, session_month, session_day, 9, 30, time_zone=DISPLAY_TZ),
            pl.datetime(session_year, session_month, session_day, 9, 30, 5, time_zone=DISPLAY_TZ),
        )
    )
    .group_by(["action", "side"])
    .agg(pl.len().alias("rows"))
    .sort(["action", "side"])
    .collect()
)

display(l2_summary)
display(l3_summary)


## Viewer

The top panel shows bid/ask price levels with L3 trade/add/modify/cancel events overlaid. L3 events are filtered to the displayed L2 depth price levels, so off-book events do not stretch the price axis. The bottom panel shows raw displayed size at each level. Use the buttons to move the current zoom window left or right by half its width.


In [ ]:
VIEW_KWARGS = dict(
    depth=DEPTH_LEVELS,
    start=START,
    end=END,
    timezone=DISPLAY_TZ,
    product=PRODUCT,
    title=f"{PRODUCT} L2 depth + L3 events | {START} to {END} {DISPLAY_TZ}",
    l3_source=l3,
    l3_depth_filter="levels",
    l3_label_cols=["flags", "order_id", "sequence", "ts_recv"],
    label_cols=["sequence"],
    include_counts=True,
    max_points=MAX_POINTS_PER_SERIES,
    template="plotly_dark",
    vertical_spacing=0.035,
)

fig = plot_order_book(book, **VIEW_KWARGS, live=True)
controls = add_x_pan_buttons(fig, step=0.5)
display(controls, fig)


In [ ]:
# The left/right buttons pan by half of the currently visible x-window.
# Use the Plotly zoom tool first, then click Left or Right to walk the window.


In [ ]:
# fig.show()


### Optional Static Export

In [ ]:
# out = PROJECT_ROOT / "notebooks" / f"{PRODUCT}_{SESSION_DATE}_order_book_view.html"
# static_fig = plot_order_book(book, **VIEW_KWARGS)
# static_fig.write_html(out)
# out
